In [ ]:
# Importe
from pathlib import Path
import gc
import json
import math
import platform
import re
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
try:
    import networkx as nx
except ImportError as exc:
    raise ImportError('networkx fehlt. Bitte in der aktivierten .venv einmal `pip install networkx` ausführen.') from exc
import pm4py
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 150)
pd.set_option('display.width', 220)
print('Python:', sys.version)
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
print('pm4py:', getattr(pm4py, '__version__', 'unknown'))
print('networkx:', nx.__version__)


In [ ]:
# Pfade und Einstellungen
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'data_raw').exists() and (candidate / 'outputs').exists():
            return candidate
    if start.name.lower() == 'notebooks':
        return start.parent
    return start
PROJECT_ROOT = find_project_root(Path.cwd())
DATA_RAW = PROJECT_ROOT / 'data_raw'
OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT = OUTPUTS_ROOT / 'heuristic_baselines_high_level_process_overview'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
PRIMARY_TARGET = 'label_scd_p90_or_global'
FINAL_PREFIX_ID = 'time120d'
FINAL_SCENARIO = 'observed_prefix'
FINAL_MODEL = 'logreg_balanced'
PREFIX_DAYS = 120.0
TRAIN_SHARE_WITHIN_2015 = 0.75
HEURISTIC_QUANTILE = 0.9
RANDOM_STATE = 42
BOOTSTRAP_REPETITIONS = 1000
MIN_EDGE_CASE_SHARE = 0.005
MAX_GRAPH_EDGES = 18
log_candidates = [DATA_RAW / 'BPI_Challenge_2018.xes.gz', *sorted(DATA_RAW.glob('**/*.xes.gz')), *sorted(DATA_RAW.glob('**/*.xes'))]
LOG_PATH = next((p for p in log_candidates if p.exists()), None)
if LOG_PATH is None:
    raise FileNotFoundError('Kein XES/XES.GZ-Log unter data_raw gefunden.')

def first_existing(paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)
RECON_CORE_PATH = first_existing([OUTPUTS_ROOT / 'benchmark_reconciliation_two_perspectives' / 'tables' / '19_case_level_reconciliation_core.csv', OUTPUTS_ROOT / 'benchmark_reconciliation_two_perspectives' / '19_case_level_reconciliation_core.csv'])
PREDICTIONS_PATH = first_existing([OUTPUTS_ROOT / 'comparative_prediction_final_robustness' / 'tables' / '07_validation_test_predictions.csv', OUTPUTS_ROOT / 'comparative_prediction_final_robustness' / '07_validation_test_predictions.csv'])
SELECTED_MODEL_PATH = first_existing([OUTPUTS_ROOT / 'comparative_prediction_final_robustness' / 'tables' / '13_selected_models_test_results.csv', OUTPUTS_ROOT / 'comparative_prediction_final_robustness' / '13_selected_models_test_results.csv'])
for name, path in {'Reconciliation Core': RECON_CORE_PATH, 'Finale Vorhersagen': PREDICTIONS_PATH, 'Finale Modellauswahl': SELECTED_MODEL_PATH}.items():
    if path is None:
        raise FileNotFoundError(f'{name} nicht gefunden. Bitte Notebook 08 und 09 vorher vollständig ausführen.')
print('Project root:', PROJECT_ROOT)
print('Log path:', LOG_PATH)
print('Reconciliation core:', RECON_CORE_PATH)
print('Predictions:', PREDICTIONS_PATH)
print('Selected model table:', SELECTED_MODEL_PATH)
print('Output root:', OUTPUT_ROOT)


In [ ]:
# Hilfsfunktionen
created_files = []
created_figures = []

def robust_to_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False).astype(bool)
    string = series.astype(str).str.strip().str.lower()
    out = pd.Series(False, index=series.index, dtype=bool)
    out[string.isin({'true', '1', 'yes', 'y', 'ja', 'wahr'})] = True
    numeric = pd.to_numeric(series, errors='coerce')
    out[numeric.fillna(0).gt(0)] = True
    return out

def save_csv(frame, filename, index=False):
    path = TABLE_DIR / filename
    frame.to_csv(path, index=index, encoding='utf-8-sig')
    created_files.append(path)
    return path

def save_json(obj, filename):
    path = TABLE_DIR / filename
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(obj, handle, indent=2, ensure_ascii=False, default=str)
    created_files.append(path)
    return path

def save_fig(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches='tight')
    plt.close(fig)
    created_files.append(path)
    created_figures.append(path)
    return path

def safe_div(num, den):
    return float(num / den) if den else np.nan

def binary_metrics(y_true, y_pred, score=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {'n': int(len(y_true)), 'positive_cases': int(y_true.sum()), 'prevalence_pct': float(y_true.mean() * 100), 'predicted_positive': int(y_pred.sum()), 'precision': float(precision_score(y_true, y_pred, zero_division=0)), 'recall': float(recall_score(y_true, y_pred, zero_division=0)), 'f1': float(f1_score(y_true, y_pred, zero_division=0)), 'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)), 'accuracy': float(accuracy_score(y_true, y_pred)), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    if score is not None and len(np.unique(y_true)) == 2:
        result['pr_auc_average_precision'] = float(average_precision_score(y_true, score))
        result['roc_auc'] = float(roc_auc_score(y_true, score))
    else:
        result['pr_auc_average_precision'] = np.nan
        result['roc_auc'] = np.nan
    return result

def per_1000_metrics(metric_row):
    n = metric_row['n']
    return {'flagged_per_1000': safe_div(metric_row['predicted_positive'] * 1000, n), 'true_positives_per_1000': safe_div(metric_row['tp'] * 1000, n), 'false_positives_per_1000': safe_div(metric_row['fp'] * 1000, n), 'false_negatives_per_1000': safe_div(metric_row['fn'] * 1000, n)}

def repeated_extra(events: pd.DataFrame, case_col: str, value_col: str, output_col: str) -> pd.DataFrame:
    if len(events) == 0:
        return pd.DataFrame(columns=[case_col, output_col])
    counts = events.groupby([case_col, value_col], observed=True).size().rename('count').reset_index()
    counts['extra'] = (counts['count'] - 1).clip(lower=0)
    return counts.groupby(case_col, observed=True)['extra'].sum().rename(output_col).reset_index()

def paired_bootstrap_difference(y_true, model_pred, heuristic_pred, repetitions=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, dtype=int)
    model_pred = np.asarray(model_pred, dtype=int)
    heuristic_pred = np.asarray(heuristic_pred, dtype=int)
    n = len(y_true)
    metrics = {'f1': [], 'precision': [], 'recall': [], 'balanced_accuracy': []}
    for _ in range(repetitions):
        idx = rng.integers(0, n, size=n)
        yt, mp, hp = (y_true[idx], model_pred[idx], heuristic_pred[idx])
        metrics['f1'].append(f1_score(yt, mp, zero_division=0) - f1_score(yt, hp, zero_division=0))
        metrics['precision'].append(precision_score(yt, mp, zero_division=0) - precision_score(yt, hp, zero_division=0))
        metrics['recall'].append(recall_score(yt, mp, zero_division=0) - recall_score(yt, hp, zero_division=0))
        metrics['balanced_accuracy'].append(balanced_accuracy_score(yt, mp) - balanced_accuracy_score(yt, hp))
    rows = []
    for metric, values in metrics.items():
        values = np.asarray(values)
        rows.append({'metric': metric, 'mean_difference_model_minus_heuristic': float(values.mean()), 'ci_lower_95': float(np.quantile(values, 0.025)), 'ci_upper_95': float(np.quantile(values, 0.975))})
    return pd.DataFrame(rows)


In [ ]:
# Event Log laden
print('Lade BPIC-2018-Log. Dieser Schritt kann mehrere Minuten dauern ...')
raw_log = pm4py.read_xes(str(LOG_PATH))
if isinstance(raw_log, pd.DataFrame):
    raw_df = raw_log
else:
    raw_df = pm4py.convert_to_dataframe(raw_log)
CASE_COL = 'case:concept:name' if 'case:concept:name' in raw_df.columns else None
TIME_COL = 'time:timestamp' if 'time:timestamp' in raw_df.columns else None
ACTIVITY_COL = 'activity' if 'activity' in raw_df.columns else 'concept:name' if 'concept:name' in raw_df.columns else None
ORDER_COL = 'identity:id' if 'identity:id' in raw_df.columns else 'eventid' if 'eventid' in raw_df.columns else None
if CASE_COL is None or TIME_COL is None or ACTIVITY_COL is None:
    raise RuntimeError(f'Kernspalten fehlen: CASE_COL={CASE_COL!r}, TIME_COL={TIME_COL!r}, ACTIVITY_COL={ACTIVITY_COL!r}')
needed_cols = [CASE_COL, TIME_COL, ACTIVITY_COL]
for col in ['doctype', 'subprocess', ORDER_COL]:
    if col is not None and col in raw_df.columns and (col not in needed_cols):
        needed_cols.append(col)
event_df = raw_df[needed_cols].copy()
del raw_log, raw_df
gc.collect()
event_df[CASE_COL] = event_df[CASE_COL].astype(str)
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce')
for col in ['doctype', 'subprocess', ACTIVITY_COL]:
    if col not in event_df.columns:
        event_df[col] = '__missing__'
    event_df[col] = event_df[col].fillna('__missing__').astype(str)
event_df['_original_row'] = np.arange(len(event_df), dtype=np.int64)
sort_cols = [CASE_COL, TIME_COL]
if ORDER_COL is not None and ORDER_COL in event_df.columns:
    event_df[ORDER_COL] = event_df[ORDER_COL].fillna('').astype(str)
    sort_cols.append(ORDER_COL)
sort_cols.append('_original_row')
event_df = event_df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
event_df['combined_activity'] = event_df['doctype'] + ' | ' + event_df['subprocess'] + ' | ' + event_df[ACTIVITY_COL]
event_df['_is_inspection_context'] = event_df['doctype'].str.contains('inspection', case=False, na=False) | event_df['subprocess'].str.contains('inspection|on-site|onsite', case=False, na=False, regex=True) | event_df['combined_activity'].str.contains('inspection|on-site|onsite', case=False, na=False, regex=True)
event_df['_case_start'] = event_df.groupby(CASE_COL, observed=True)[TIME_COL].transform('min')
event_df['_case_end'] = event_df.groupby(CASE_COL, observed=True)[TIME_COL].transform('max')
event_df['_elapsed_days'] = (event_df[TIME_COL] - event_df['_case_start']).dt.total_seconds() / 86400
basic_info = {'events': int(len(event_df)), 'cases': int(event_df[CASE_COL].nunique()), 'timestamp_min': str(event_df[TIME_COL].min()), 'timestamp_max': str(event_df[TIME_COL].max()), 'prefix_days': PREFIX_DAYS, 'tie_policy_process_graph': 'events at identical timestamps are treated as an unordered bucket'}
save_json(basic_info, '00_basic_info.json')
print(json.dumps(basic_info, indent=2, ensure_ascii=False))


In [ ]:
# Falldaten und Modell laden
core = pd.read_csv(RECON_CORE_PATH, dtype={CASE_COL: str})
core[CASE_COL] = core[CASE_COL].astype(str)
if PRIMARY_TARGET not in core.columns:
    raise RuntimeError(f'Primary target fehlt im Reconciliation Core: {PRIMARY_TARGET}')
core[PRIMARY_TARGET] = robust_to_bool(core[PRIMARY_TARGET])
for col in [c for c in core.columns if c.startswith('has_') or c.startswith('selected_')]:
    core[col] = robust_to_bool(core[col])
core['case_year'] = core['case_year'].astype(str)
case_times = event_df.groupby(CASE_COL, observed=True).agg(case_start=(TIME_COL, 'min'), case_end=(TIME_COL, 'max')).reset_index()
case_times['duration_days_recomputed'] = (case_times['case_end'] - case_times['case_start']).dt.total_seconds() / 86400
core = core.merge(case_times, on=CASE_COL, how='left', validate='one_to_one')
core = core.sort_values(['case_year', 'case_start', CASE_COL], kind='mergesort').reset_index(drop=True)
core['split'] = 'unused'
cases_2015 = core[core['case_year'].eq('2015')].sort_values(['case_start', CASE_COL], kind='mergesort')
cut = int(len(cases_2015) * TRAIN_SHARE_WITHIN_2015)
train_ids = set(cases_2015.iloc[:cut][CASE_COL])
validation_ids = set(cases_2015.iloc[cut:][CASE_COL])
test_ids = set(core.loc[core['case_year'].eq('2016'), CASE_COL])
sensitivity_ids = set(core.loc[core['case_year'].eq('2017'), CASE_COL])
core.loc[core[CASE_COL].isin(train_ids), 'split'] = 'train'
core.loc[core[CASE_COL].isin(validation_ids), 'split'] = 'validation'
core.loc[core[CASE_COL].isin(test_ids), 'split'] = 'test'
core.loc[core[CASE_COL].isin(sensitivity_ids), 'split'] = 'sensitivity2017'
split_registry = core.groupby('split', observed=True).agg(n_cases=(CASE_COL, 'size'), positive_cases=(PRIMARY_TARGET, 'sum'), prevalence=(PRIMARY_TARGET, 'mean')).reset_index()
split_registry['prevalence_pct'] = split_registry['prevalence'] * 100
save_csv(split_registry, '01_reproduced_split_registry.csv')
selected_models = pd.read_csv(SELECTED_MODEL_PATH)
final_selected = selected_models[selected_models['target'].eq(PRIMARY_TARGET) & selected_models['prefix_id'].eq(FINAL_PREFIX_ID) & selected_models['scenario'].eq(FINAL_SCENARIO) & selected_models['model'].eq(FINAL_MODEL) & selected_models['split'].eq('test')].copy()
if len(final_selected) != 1:
    raise RuntimeError(f'Finale Modellzeile nicht eindeutig gefunden: {len(final_selected)} Zeilen')
final_model_threshold = float(final_selected.iloc[0]['validation_selected_threshold'])
prediction_parts = []
usecols = [CASE_COL, 'split', 'target', 'prefix_id', 'scenario', 'model', 'y_true', 'score', 'threshold', 'pred_validation_threshold']
for chunk in pd.read_csv(PREDICTIONS_PATH, usecols=usecols, chunksize=250000, low_memory=False):
    mask = chunk['target'].eq(PRIMARY_TARGET) & chunk['prefix_id'].eq(FINAL_PREFIX_ID) & chunk['scenario'].eq(FINAL_SCENARIO) & chunk['model'].eq(FINAL_MODEL) & chunk['split'].isin(['validation', 'test'])
    if mask.any():
        prediction_parts.append(chunk.loc[mask].copy())
model_predictions = pd.concat(prediction_parts, ignore_index=True)
model_predictions[CASE_COL] = model_predictions[CASE_COL].astype(str)
model_predictions['model_pred'] = pd.to_numeric(model_predictions['pred_validation_threshold'], errors='coerce').fillna(0).astype(int)
model_predictions['score'] = pd.to_numeric(model_predictions['score'], errors='coerce')
model_predictions['y_true'] = pd.to_numeric(model_predictions['y_true'], errors='coerce').astype(int)
expected_counts = {'validation': len(validation_ids), 'test': len(test_ids)}
actual_counts = model_predictions.groupby('split')[CASE_COL].nunique().to_dict()
if any((actual_counts.get(split, 0) != expected for split, expected in expected_counts.items())):
    raise RuntimeError(f'Modellvorhersagen passen nicht zum reproduzierten Split: {actual_counts} vs. {expected_counts}')
display(split_registry)
display(final_selected)
print('Finale Vorhersagen geladen:', model_predictions.shape)


In [ ]:
# Präfixmerkmale für Heuristiken
prefix_events = event_df[event_df['_elapsed_days'].le(PREFIX_DAYS)].copy()
print('Events im 120-Tage-Prefix:', len(prefix_events))
prefix_counts = prefix_events.groupby(CASE_COL, observed=True).size().rename('prefix_event_count').reset_index()
prefix_rework = repeated_extra(prefix_events, CASE_COL, 'combined_activity', 'prefix_combined_rework_extra')
prefix_inspection = prefix_events.groupby(CASE_COL, observed=True)['_is_inspection_context'].any().rename('prefix_has_inspection_context').reset_index()
prefix_last = prefix_events.groupby(CASE_COL, observed=True)[TIME_COL].max().rename('prefix_last_event_time').reset_index()
heuristic_df = core[[CASE_COL, 'split', PRIMARY_TARGET, 'case_year', 'case_department', 'has_inspection_event_context', 'duration_days_recomputed']].copy()
heuristic_df = heuristic_df.merge(prefix_counts, on=CASE_COL, how='left')
heuristic_df = heuristic_df.merge(prefix_rework, on=CASE_COL, how='left')
heuristic_df = heuristic_df.merge(prefix_inspection, on=CASE_COL, how='left')
heuristic_df = heuristic_df.merge(prefix_last, on=CASE_COL, how='left')
heuristic_df['prefix_event_count'] = heuristic_df['prefix_event_count'].fillna(0).astype(int)
heuristic_df['prefix_combined_rework_extra'] = heuristic_df['prefix_combined_rework_extra'].fillna(0).astype(int)
heuristic_df['prefix_has_inspection_context'] = robust_to_bool(heuristic_df['prefix_has_inspection_context'])
heuristic_df['has_inspection_event_context'] = robust_to_bool(heuristic_df['has_inspection_event_context'])
heuristic_df[PRIMARY_TARGET] = robust_to_bool(heuristic_df[PRIMARY_TARGET])
train_prefix = heuristic_df[heuristic_df['split'].eq('train')]
threshold_event = float(train_prefix['prefix_event_count'].quantile(HEURISTIC_QUANTILE))
threshold_rework = float(train_prefix['prefix_combined_rework_extra'].quantile(HEURISTIC_QUANTILE))
threshold_registry = pd.DataFrame([{'heuristic_component': 'prefix_event_count', 'quantile': HEURISTIC_QUANTILE, 'threshold': threshold_event, 'source_population': 'train only — earliest 75% of 2015 cases'}, {'heuristic_component': 'prefix_combined_rework_extra', 'quantile': HEURISTIC_QUANTILE, 'threshold': threshold_rework, 'source_population': 'train only — earliest 75% of 2015 cases'}])
save_csv(threshold_registry, '02_heuristic_threshold_registry.csv')
heuristic_df['heuristic_inspection'] = heuristic_df['prefix_has_inspection_context']
heuristic_df['heuristic_early_load'] = heuristic_df['prefix_event_count'].ge(threshold_event) | heuristic_df['prefix_combined_rework_extra'].ge(threshold_rework)
heuristic_df['heuristic_combined'] = heuristic_df['heuristic_inspection'] | heuristic_df['heuristic_early_load']
heuristic_df['early_load_index'] = np.maximum(heuristic_df['prefix_event_count'] / max(threshold_event, 1e-09), heuristic_df['prefix_combined_rework_extra'] / max(threshold_rework, 1e-09))
save_csv(heuristic_df[[CASE_COL, 'split', PRIMARY_TARGET, 'prefix_event_count', 'prefix_combined_rework_extra', 'prefix_has_inspection_context', 'heuristic_inspection', 'heuristic_early_load', 'heuristic_combined', 'early_load_index', 'has_inspection_event_context', 'duration_days_recomputed']], '03_case_level_heuristic_features_and_predictions.csv')
display(threshold_registry)
display(heuristic_df.head())


In [ ]:
# Modell und Heuristiken vergleichen
comparison_df = heuristic_df.merge(model_predictions[[CASE_COL, 'split', 'y_true', 'score', 'model_pred']], on=[CASE_COL, 'split'], how='inner', validate='one_to_one')
if not np.array_equal(comparison_df[PRIMARY_TARGET].astype(int).to_numpy(), comparison_df['y_true'].astype(int).to_numpy()):
    raise RuntimeError('Target aus Core und Target aus Prediction-Datei stimmen nicht überein.')
prediction_columns = {'Final Logistic Regression': ('model_pred', 'score'), 'Inspection heuristic': ('heuristic_inspection', None), 'Early-load heuristic': ('heuristic_early_load', 'early_load_index'), 'Combined heuristic': ('heuristic_combined', None)}
metric_rows = []
workload_rows = []
for split in ['validation', 'test']:
    sub = comparison_df[comparison_df['split'].eq(split)].copy()
    y_true = sub['y_true'].astype(int)
    for method, (pred_col, score_col) in prediction_columns.items():
        y_pred = robust_to_bool(sub[pred_col]).astype(int)
        score = sub[score_col] if score_col is not None else None
        metrics = binary_metrics(y_true, y_pred, score=score)
        metric_rows.append({'split': split, 'method': method, **metrics})
        workload_rows.append({'split': split, 'method': method, **per_1000_metrics(metrics)})
metrics_df = pd.DataFrame(metric_rows)
workload_df = pd.DataFrame(workload_rows)
save_csv(metrics_df, '04_model_vs_heuristics_metrics.csv')
save_csv(workload_df, '05_model_vs_heuristics_workload_per_1000.csv')
sub_test = comparison_df[comparison_df['split'].eq('test')].copy()
y_test = sub_test['y_true'].astype(int)
reclassification_rows = []
bootstrap_parts = []
for heuristic_name, pred_col in [('Inspection heuristic', 'heuristic_inspection'), ('Early-load heuristic', 'heuristic_early_load'), ('Combined heuristic', 'heuristic_combined')]:
    hpred = robust_to_bool(sub_test[pred_col]).astype(int)
    mpred = sub_test['model_pred'].astype(int)
    reclassification_rows.append({'heuristic': heuristic_name, 'both_flagged': int(((mpred == 1) & (hpred == 1)).sum()), 'model_only_flagged': int(((mpred == 1) & (hpred == 0)).sum()), 'heuristic_only_flagged': int(((mpred == 0) & (hpred == 1)).sum()), 'neither_flagged': int(((mpred == 0) & (hpred == 0)).sum()), 'model_only_true_positives': int(((mpred == 1) & (hpred == 0) & (y_test == 1)).sum()), 'heuristic_only_true_positives': int(((mpred == 0) & (hpred == 1) & (y_test == 1)).sum()), 'model_only_false_positives': int(((mpred == 1) & (hpred == 0) & (y_test == 0)).sum()), 'heuristic_only_false_positives': int(((mpred == 0) & (hpred == 1) & (y_test == 0)).sum())})
    boot = paired_bootstrap_difference(y_test, mpred, hpred, repetitions=BOOTSTRAP_REPETITIONS, seed=RANDOM_STATE)
    boot.insert(0, 'heuristic', heuristic_name)
    bootstrap_parts.append(boot)
reclassification_df = pd.DataFrame(reclassification_rows)
bootstrap_df = pd.concat(bootstrap_parts, ignore_index=True)
save_csv(reclassification_df, '06_reclassification_model_vs_heuristics.csv')
save_csv(bootstrap_df, '07_paired_bootstrap_metric_differences.csv')
subgroup_rows = []
for group_col in ['has_inspection_event_context', 'prefix_has_inspection_context']:
    for group_value in [False, True]:
        group = sub_test[robust_to_bool(sub_test[group_col]).eq(group_value)]
        if len(group) == 0:
            continue
        for method, (pred_col, score_col) in prediction_columns.items():
            metrics = binary_metrics(group['y_true'].astype(int), robust_to_bool(group[pred_col]).astype(int), score=group[score_col] if score_col is not None else None)
            subgroup_rows.append({'group_variable': group_col, 'group_value': group_value, 'method': method, **metrics})
subgroup_df = pd.DataFrame(subgroup_rows)
save_csv(subgroup_df, '08_inspection_subgroup_model_vs_heuristics.csv')
display(metrics_df[metrics_df['split'].eq('test')].sort_values('f1', ascending=False))
display(reclassification_df)
display(bootstrap_df)


In [ ]:
# Heuristikvergleich abbilden
test_metrics = metrics_df[metrics_df['split'].eq('test')].copy()
plot_metrics = ['precision', 'recall', 'f1', 'balanced_accuracy']
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(test_metrics))
width = 0.18
for idx, metric in enumerate(plot_metrics):
    ax.bar(x + (idx - 1.5) * width, test_metrics[metric], width=width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(test_metrics['method'], rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Score auf dem 2016er Testsplit')
ax.set_title('Finales Modell und transparente Heuristik-Baselines')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_01_model_vs_heuristics_metrics.png')
work_test = workload_df[workload_df['split'].eq('test')].copy()
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(work_test))
width = 0.24
for idx, metric in enumerate(['true_positives_per_1000', 'false_positives_per_1000', 'false_negatives_per_1000']):
    ax.bar(x + (idx - 1) * width, work_test[metric], width=width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(work_test['method'], rotation=15, ha='right')
ax.set_ylabel('Fälle je 1.000 Testfälle')
ax.set_title('Operative Konsequenz der unterschiedlichen Früherkennungsregeln')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_02_operational_workload_per_1000.png')
recall_group = subgroup_df[subgroup_df['group_variable'].eq('has_inspection_event_context')].copy()
pivot = recall_group.pivot(index='method', columns='group_value', values='recall').reset_index()
pivot = pivot.rename(columns={False: 'ohne Inspection', True: 'mit Inspection'})
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(pivot))
ax.bar(x - 0.18, pivot.get('ohne Inspection', np.nan), width=0.36, label='ohne Inspection')
ax.bar(x + 0.18, pivot.get('mit Inspection', np.nan), width=0.36, label='mit Inspection')
ax.set_xticks(x)
ax.set_xticklabels(pivot['method'], rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Recall')
ax.set_title('Erkennungsleistung nach späterem Inspection-Kontext')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_03_recall_by_inspection_context.png')


In [ ]:
# Prozessübersicht
sub_event = event_df[[CASE_COL, TIME_COL, 'subprocess', '_case_start', '_case_end']].copy()
sub_event = sub_event[sub_event['subprocess'].ne('__missing__')]
sub_event = sub_event.dropna(subset=[TIME_COL])
case_total = event_df[CASE_COL].nunique()
node_presence = sub_event[[CASE_COL, 'subprocess']].drop_duplicates().groupby('subprocess', observed=True)[CASE_COL].nunique().rename('n_cases').reset_index()
node_presence['case_share_pct'] = node_presence['n_cases'] / case_total * 100
first_sub = sub_event.groupby([CASE_COL, 'subprocess'], observed=True)[TIME_COL].min().rename('first_time').reset_index()
case_bounds = sub_event.groupby(CASE_COL, observed=True)[TIME_COL].agg(case_start='min', case_end='max').reset_index()
first_sub = first_sub.merge(case_bounds, on=CASE_COL, how='left')
denominator = (first_sub['case_end'] - first_sub['case_start']).dt.total_seconds()
numerator = (first_sub['first_time'] - first_sub['case_start']).dt.total_seconds()
first_sub['relative_first_position'] = np.where(denominator.gt(0), numerator / denominator, 0.0)
first_sub['first_elapsed_days'] = numerator / 86400
node_timing = first_sub.groupby('subprocess', observed=True).agg(median_relative_first_position=('relative_first_position', 'median'), median_first_elapsed_days=('first_elapsed_days', 'median')).reset_index()
node_summary = node_presence.merge(node_timing, on='subprocess', how='left')
node_summary = node_summary.sort_values('median_relative_first_position')
save_csv(node_summary, '09_subprocess_node_summary.csv')
bucket_source = sub_event[[CASE_COL, TIME_COL, 'subprocess']].drop_duplicates()
buckets = bucket_source.groupby([CASE_COL, TIME_COL], observed=True)['subprocess'].agg(lambda values: tuple(sorted(set(values)))).rename('source_set').reset_index().sort_values([CASE_COL, TIME_COL], kind='mergesort')
buckets['previous_set'] = buckets.groupby(CASE_COL, observed=True)['source_set'].shift(1)
buckets = buckets[buckets['source_set'].ne(buckets['previous_set'])].copy()
buckets['target_set'] = buckets.groupby(CASE_COL, observed=True)['source_set'].shift(-1)
buckets = buckets[buckets['target_set'].notna()].copy()
buckets['source_n'] = buckets['source_set'].map(len)
buckets['target_n'] = buckets['target_set'].map(len)
transitions = buckets[[CASE_COL, 'source_set', 'target_set', 'source_n', 'target_n']].copy()
transitions = transitions.explode('source_set').rename(columns={'source_set': 'source'})
transitions = transitions.explode('target_set').rename(columns={'target_set': 'target'})
transitions = transitions[transitions['source'].ne(transitions['target'])].copy()
transitions['bucket_weight'] = 1.0 / (transitions['source_n'] * transitions['target_n'])
edge_weight = transitions.groupby(['source', 'target'], observed=True)['bucket_weight'].sum().rename('weighted_transition_count')
edge_cases = transitions[[CASE_COL, 'source', 'target']].drop_duplicates().groupby(['source', 'target'], observed=True)[CASE_COL].nunique().rename('n_cases')
edge_summary = pd.concat([edge_weight, edge_cases], axis=1).reset_index()
edge_summary['case_share_pct'] = edge_summary['n_cases'] / case_total * 100
edge_summary = edge_summary.sort_values(['n_cases', 'weighted_transition_count'], ascending=False)
save_csv(edge_summary, '10_subprocess_transition_summary_all.csv')
min_cases = max(50, int(math.ceil(case_total * MIN_EDGE_CASE_SHARE)))
graph_edges = edge_summary[edge_summary['n_cases'].ge(min_cases)].head(MAX_GRAPH_EDGES).copy()
save_csv(graph_edges, '11_subprocess_transition_summary_graph_edges.csv')
node_lookup = node_summary.set_index('subprocess')
active_nodes = sorted(set(graph_edges['source']) | set(graph_edges['target']))
active_nodes = [n for n in active_nodes if n in node_lookup.index]
active_nodes_sorted = sorted(active_nodes, key=lambda n: node_lookup.loc[n, 'median_relative_first_position'])
semantic_y = {'Application': 0.0, 'Main': 0.15, 'Declared': -0.75, 'Reported': 0.75, 'Remote': -1.25, 'On-Site': 1.25, 'Change': 0.85, 'Objection': -0.85}
pos = {}
for idx, node in enumerate(active_nodes_sorted):
    x = float(node_lookup.loc[node, 'median_relative_first_position'])
    y = semantic_y.get(node, (idx % 5 - 2) * 0.45)
    pos[node] = (x, y)
G = nx.DiGraph()
for node in active_nodes:
    G.add_node(node)
for _, row in graph_edges.iterrows():
    if row['source'] in active_nodes and row['target'] in active_nodes:
        G.add_edge(row['source'], row['target'], case_share_pct=float(row['case_share_pct']))
fig, ax = plt.subplots(figsize=(13, 8))
node_sizes = [900 + 42 * float(node_lookup.loc[node, 'case_share_pct']) for node in G.nodes()]
edge_values = [G[u][v]['case_share_pct'] for u, v in G.edges()]
max_edge = max(edge_values) if edge_values else 1
edge_widths = [0.8 + 5.0 * value / max_edge for value in edge_values]
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='white', edgecolors='black', linewidths=1.2, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='black', alpha=0.55, arrows=True, arrowsize=16, connectionstyle='arc3,rad=0.08', ax=ax)
node_labels = {node: f"{node}\n{node_lookup.loc[node, 'case_share_pct']:.1f}% der Cases" for node in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=9, ax=ax)
label_edges = graph_edges.head(10)
edge_labels = {(row['source'], row['target']): f"{row['case_share_pct']:.1f}%" for _, row in label_edges.iterrows() if G.has_edge(row['source'], row['target'])}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, rotate=False, ax=ax)
ax.set_title('High-Level-Prozessübersicht auf Subprocess-Ebene')
ax.text(0.5, -0.08, 'Knotenposition x: medianer relativer Zeitpunkt des ersten Auftretens. Übergänge: aufeinanderfolgende unterschiedliche Timestamp-Buckets; keine Reihenfolge innerhalb identischer Timestamps.', ha='center', va='top', transform=ax.transAxes, fontsize=9)
ax.set_axis_off()
save_fig(fig, 'fig_04_high_level_subprocess_overview.png')
display(node_summary)
display(graph_edges)


In [ ]:
# Ergebnisse speichern
test_table = metrics_df[metrics_df['split'].eq('test')].copy().sort_values('f1', ascending=False)
thesis_table = test_table[['method', 'n', 'positive_cases', 'prevalence_pct', 'predicted_positive', 'precision', 'recall', 'f1', 'balanced_accuracy', 'tp', 'fp', 'fn', 'tn']].copy()
save_csv(thesis_table, '15_thesis_table_model_vs_heuristics.csv')
display(thesis_table)
